# 04 - Huấn Luyện Mô Hình Spatiotemporal ConvLSTM
### Dự án: Dự Báo Lượng Mưa Cực Ngắn (Rainfall Nowcasting) tại Việt Nam
---

Notebook này xây dựng và huấn luyện kiến trúc mạng **Convolutional LSTM (ConvLSTM)** cho bài toán Spatiotemporal Nowcasting:
- **Đầu vào $X$:** 6 frames quá khứ (3 giờ hoặc 6 giờ) gồm 5 kênh khí tượng ($tp, t_{2m}, msl, u_{10}, v_{10}$).
- **Đầu ra $Y$:** 6 frames tương lai (lượng mưa $tp$).


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Import Dataset loader từ dataset.py
from dataset import load_processed_loaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị tính toán đang sử dụng: {device}")


## 1. Nạp Dữ liệu với PyTorch DataLoader Tối ưu Bộ nhớ


In [ ]:
# Nạp dữ liệu với batch size phù hợp
train_loader, val_loader, test_loader, metadata = load_processed_loaders(
    processed_dir="processed_data",
    batch_size=8,
    in_steps=6,
    out_steps=6,
    filter_dry_samples=True  # Lọc mẫu có mưa để tăng cường học các đợt mưa bão
)

print(f"Số lượng batch Train: {len(train_loader):,}")
print(f"Số lượng batch Val:   {len(val_loader):,}")
print(f"Số lượng batch Test:  {len(test_loader):,}")


## 2. Định nghĩa Kiến trúc Mạng ConvLSTM (Encoder - Decoder)


In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=(3, 3), bias=True):
        super(ConvLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.conv = nn.Conv2d(
            in_channels=input_dim + hidden_dim,
            out_channels=4 * hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            bias=bias
        )

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class ConvLSTMNowcastNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=1, hidden_dim=32, out_steps=6):
        super(ConvLSTMNowcastNet, self).__init__()
        self.out_steps = out_steps
        self.hidden_dim = hidden_dim
        
        self.encoder = ConvLSTMCell(input_dim=in_channels, hidden_dim=hidden_dim)
        self.decoder = ConvLSTMCell(input_dim=hidden_dim, hidden_dim=hidden_dim)
        self.out_conv = nn.Sequential(
            nn.Conv2d(hidden_dim, 16, kernel_size=3, padding=1),
            nn.LeakyReLU(0.1),
            nn.Conv2d(16, out_channels, kernel_size=1),
            nn.ReLU() # Lượng mưa luôn không âm
        )

    def forward(self, x):
        b, seq_len, _, h, w = x.size()
        h_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        c_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        
        # 1. Encoding giai đoạn quá khứ
        for t in range(seq_len):
            h_enc, c_enc = self.encoder(x[:, t], (h_enc, c_enc))
            
        # 2. Decoding giai đoạn tương lai
        h_dec, c_dec = h_enc, c_enc
        outputs = []
        for t in range(self.out_steps):
            h_dec, c_dec = self.decoder(h_dec, (h_dec, c_dec))
            out = self.out_conv(h_dec)
            outputs.append(out)
            
        return torch.stack(outputs, dim=1)

# Khởi tạo mô hình (5 kênh đầu vào, 1 kênh lượng mưa đầu ra, dự báo 6 frames)
model = ConvLSTMNowcastNet(in_channels=5, out_channels=1, hidden_dim=32, out_steps=6).to(device)
print(f"Tổng số tham số của mô hình: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 3. Huấn luyện Mô hình với Balanced Loss


In [ ]:
# Hàm Loss Balanced MSE tăng trọng số cho các điểm ảnh có mưa to
class BalancedMSELoss(nn.Module):
    def __init__(self, rain_weight=5.0):
        super(BalancedMSELoss, self).__init__()
        self.rain_weight = rain_weight
        
    def forward(self, y_pred, y_true):
        # Trọng số w = 1 nếu y_true = 0, w = rain_weight nếu y_true > 0
        weights = 1.0 + (self.rain_weight - 1.0) * (y_true > 0.05).float()
        loss = weights * (y_pred - y_true) ** 2
        return torch.mean(loss)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
criterion = BalancedMSELoss(rain_weight=5.0)

EPOCHS = 10
best_val_loss = float('inf')

print("Bắt đầu huấn luyện ConvLSTM...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    
    # Validation loop
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_x, val_y in val_loader:
            val_x, val_y = val_x.to(device), val_y.to(device)
            preds = model(val_x)
            val_loss += criterion(preds, val_y).item()
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'convlstm_best_model.pth')
        print(f"-> Đã lưu best checkpoint (Val Loss: {best_val_loss:.6f})")

print("Hoàn tất huấn luyện ConvLSTM!")
